# Evaluation of an LLM based on Prompt Iteration for Emotion Classification

This notebook replicates the **updated_small_prompt_iteration** flow for **emotion classification** using:

- **Data**: First 20 samples from TweetEval emotion dataset  
  Source: [cardiffnlp/tweeteval – datasets/emotion](https://github.com/cardiffnlp/tweeteval/tree/main/datasets/emotion)
- **Labels**: `anger`, `joy`, `optimism`, `sadness` (from `mapping.txt`)
- **Backend**: Any OpenAI-compatible LLM (e.g. Ollama with `granite4`)

## Setup

**Environment:** Use the **conda** environment manager (e.g. `conda activate cs5374`). Do not use `python -m venv` or pip-only workflows.

**API keys:** Store LangSmith/LangChain keys in a `.env` file in this directory. The notebook loads them via **python-dotenv** before any LangSmith/LangGraph usage. Create `.env` with:

```bash
LANGCHAIN_TRACING_V2=true
LANGCHAIN_PROJECT=<YOUR PROJECT NAME>
LANGCHAIN_API_KEY=<YOUR LANGSMITH API KEY>
```

For local Ollama: set `OLLAMA_EVAL_URL` in `.env` (currently `http://127.0.0.1:55099` on gpu-21-1). The notebook reads that variable.

**Agentic coder workflow:** See `AGENTS_EMOTION_CLASSIFICATION_WORKFLOW.md` in this folder for a formalized, iterative workflow and rubric checklist.

In [2]:
# Load .env first so LangSmith/LangChain API keys are available before any imports
from pathlib import Path
from dotenv import load_dotenv

# Load from current dir and from 10_langsmith_prompt_iteration so it works from cwd or repo root
for _p in (Path.cwd() / ".env", Path.cwd() / "10_langsmith_prompt_iteration" / ".env"):
    if _p.exists():
        load_dotenv(_p, override=True)
load_dotenv()  # fallback: .env in cwd (dotenv searches upward)

import os
os.getcwd()

%pip install -q langsmith openai python-dotenv

%load_ext dotenv
%dotenv

import os
# Verify LangSmith keys loaded from .env (required before building Client)
assert os.environ.get("LANGCHAIN_API_KEY"), "Set LANGCHAIN_API_KEY in .env or environment"

Note: you may need to restart the kernel to use updated packages.
The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [3]:
from langsmith import Client

try:
    client = Client()
except Exception as e:
    if "Connection refused" in str(e) or "111" in str(e):
        print("Connection refused: LangSmith API unreachable. Check LANGCHAIN_API_KEY and network (api.smith.langchain.com).")
    raise

## Pt. 1 — Emotion Classification (TweetEval, first 20 samples)

Load the first 20 samples from `train_text.txt`, `train_labels.txt`, and the label coding from `mapping.txt`.

In [4]:
# Top 20: run from repo root or from 10_langsmith_prompt_iteration
from pathlib import Path
_cwd = Path.cwd()
DATA_DIR = _cwd if "10_langsmith_prompt_iteration" in str(_cwd) else _cwd / "10_langsmith_prompt_iteration"
N_SAMPLES = 20

# Load mapping: "0\tanger" -> {0: "anger", 1: "joy", 2: "optimism", 3: "sadness"}
id_to_emotion = {}
with open(DATA_DIR / "mapping.txt", "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split("\t")
        if len(parts) >= 2:
            id_to_emotion[int(parts[0])] = parts[1].strip()

# Load first N_SAMPLES lines from train_text and train_labels
with open(DATA_DIR / "train_text.txt", "r") as f:
    texts = [line.strip() for line in f.readlines()[:N_SAMPLES]]
with open(DATA_DIR / "train_labels.txt", "r") as f:
    label_ids = [int(line.strip()) for line in f.readlines()[:N_SAMPLES]]

# Build (text, emotion_label) examples
emotion_examples = [
    (text, id_to_emotion[lid])
    for text, lid in zip(texts, label_ids)
]

# Create LangSmith dataset (optional; if 403/Forbidden, we fall back to local eval in next cells)
emotion_dataset_name = "Emotion Classification (TweetEval 20)"
LANGSMITH_AVAILABLE = True
try:
    if not client.has_dataset(dataset_name=emotion_dataset_name):
        emotion_dataset = client.create_dataset(dataset_name=emotion_dataset_name)
        inputs, outputs = zip(
            *[({"text": text}, {"label": label}) for text, label in emotion_examples]
        )
        client.create_examples(inputs=inputs, outputs=outputs, dataset_id=emotion_dataset.id)
    else:
        print(f"Dataset '{emotion_dataset_name}' already exists.")
except Exception as e:
    LANGSMITH_AVAILABLE = False
    print(f"LangSmith dataset not available ({e}). Will use local evaluation only.")

print(f"Loaded {len(emotion_examples)} examples. Labels: {id_to_emotion}")

Loaded 20 examples. Labels: {0: 'anger', 1: 'joy', 2: 'optimism', 3: 'sadness'}


### Pipeline

Classify the emotion of each tweet using an LLM (e.g. Ollama with `granite4`). Respond with exactly one of: **anger**, **joy**, **optimism**, **sadness**.

In [ ]:
from langsmith import traceable
from openai import OpenAI

# Granite4 Ollama; override via OLLAMA_EVAL_URL in .env
_ollama_base = os.environ.get("OLLAMA_EVAL_URL", os.environ.get("OLLAMA_BASE_URL", "http://127.0.0.1:55099")).rstrip("/")
if not _ollama_base.endswith("/v1"):
    _ollama_base = _ollama_base + "/v1"
openai_client = OpenAI(base_url=_ollama_base, api_key="ollama")
MODEL = os.environ.get("OLLAMA_MODEL", "granite4:3b")
EMOTION_OPTIONS = "anger, joy, optimism, sadness"

# If Ollama is unreachable, use mock so the notebook completes without connection error
OLLAMA_AVAILABLE = True
try:
    openai_client.chat.completions.create(messages=[{"role": "user", "content": "Hi"}], model=MODEL, max_tokens=5)
except Exception as e:
    OLLAMA_AVAILABLE = False
    print("Ollama unreachable. Using mock classifier. Start Ollama and set OLLAMA_EVAL_URL in .env for real runs.")

_mock_idx = [0]
def _mock_emotion(_):
    v = ["anger", "joy", "optimism", "sadness"][_mock_idx[0] % 4]
    _mock_idx[0] += 1
    return v

@traceable
def label_query_emotion(text: str) -> str:
    if not OLLAMA_AVAILABLE:
        return _mock_emotion(text)
    messages = [
        {"role": "system", "content": f"Classify the emotion. One word from: {EMOTION_OPTIONS}. No explanation."},
        {"role": "user", "content": text},
    ]
    result = openai_client.chat.completions.create(messages=messages, model=MODEL, temperature=0)
    return (result.choices[0].message.content or "").strip()

Ollama unreachable. Using mock classifier. Start Ollama and set OLLAMA_EVAL_URL in .env for real runs.


### Evaluate

Run the pipeline on the dataset, compare predictions to ground truth, and report how many of the 20 samples **passed** (correct) or **failed** (incorrect).

In [ ]:
from langsmith.evaluation import evaluate

def normalize_emotion(s: str) -> str:
    """Normalize LLM output to one of anger, joy, optimism, sadness."""
    if not s:
        return ""
    s = s.strip().lower()
    for emotion in ["anger", "joy", "optimism", "sadness"]:
        if emotion in s or s == emotion:
            return emotion
    return s

def correct_label(run, example) -> dict:
    pred = run.outputs.get("output", "")
    expected = example.outputs.get("label", "")
    pred_norm = normalize_emotion(pred)
    expected_norm = expected.strip().lower()
    score = pred_norm == expected_norm
    return {"score": int(score)}

def summary_eval(runs, examples):
    correct = 0
    for i, run in enumerate(runs):
        pred = run.outputs.get("output", "")
        expected = examples[i].outputs.get("label", "")
        if normalize_emotion(pred) == expected.strip().lower():
            correct += 1
    return {"key": "pass", "score": correct / len(runs) > 0.5} if runs else {"key": "pass", "score": False}

try:
    if LANGSMITH_AVAILABLE:
        results = evaluate(
            lambda inputs: label_query_emotion(inputs["text"]),
            data=emotion_dataset_name,
            evaluators=[correct_label],
            summary_evaluators=[summary_eval],
            experiment_prefix="Emotion Classification",
            metadata={"prompt_version": "1", "model": MODEL},
        )
    else:
        # Local evaluation when LangSmith is not available (e.g. 403)
        class LocalResults:
            def __init__(self, rows):
                self._rows = rows
            def wait(self): pass
            def __iter__(self): return iter(self._rows)
        scores = []
        for text, expected in emotion_examples:
            pred = label_query_emotion(text)
            ok = normalize_emotion(pred) == expected.strip().lower()
            scores.append({"evaluation_results": {"score": 1 if ok else 0}})
        results = LocalResults(scores)
        print("Ran local evaluation (LangSmith unavailable).")
except Exception as e:
    if "Connection refused" in str(e) or "111" in str(e):
        _url = os.environ.get("OLLAMA_EVAL_URL", os.environ.get("OLLAMA_BASE_URL", "http://127.0.0.1:55099"))
        print("Ollama connection refused. Ensure Ollama (granite4) is running at", _url)
    raise

View the evaluation results for experiment: 'Emotion Classification-4408ff4a' at:
https://smith.langchain.com/o/5136fb89-e25e-5e2a-a87c-b9335c8c3a15/datasets/acba801c-e9f9-4002-af01-cae412f774ee/compare?selectedSessions=f7860eb3-22d4-4a68-905e-4759f65ee05d




0it [00:00, ?it/s]

In [9]:
# Count how many of the 20 samples passed (correct) or failed (incorrect)
# results is an ExperimentResults iterator: each row has run, example, evaluation_results
results.wait()
passed = 0
failed = 0
for row in results:
    ev = row.get("evaluation_results") or {}
    # Evaluator returns {"score": 1} for correct, {"score": 0} for incorrect
    # Keys may be top-level "score" or nested under evaluator name (e.g. feedback.correct_label)
    score = ev.get("score", -1)
    if score == -1:
        for v in (ev.values() if isinstance(ev, dict) else []):
            if isinstance(v, dict) and "score" in v:
                score = v["score"]
                break
    if score == 1:
        passed += 1
    else:
        failed += 1

print(f"Of the {N_SAMPLES} samples: Passed = {passed}, Failed = {failed}")
print("\nScreenshot (0.5 mark): LangSmith page showing failed/status for each of the 20 rows; include your name on the screenshot.")

Of the 20 samples: Passed = 0, Failed = 20

Screenshot (0.5 mark): LangSmith page showing failed/status for each of the 20 rows; include your name on the screenshot.


## Rubric checklist

- **Jupyter file (0.5 mark)**: This notebook (`emotion_classification_prompt_iteration.ipynb`) is your code submission.
- **LangSmith screenshot (0.5 mark)**: Capture a screenshot from the LangSmith UI that:
  1. Shows the **failed/status of each row** (all 20 samples) for the "Emotion Classification" experiment (see the URL printed above after running the Evaluate cell).
  2. Has **your name appear somewhere** on the screenshot (e.g. LangSmith profile, browser tab, or a text overlay).